# 🛠️ 02 - Preprocessing & Feature Engineering
Notebook này thực hiện:
1. Xử lý giá trị thiếu (Missing values) và ngoại lai (Outliers)
2. Tạo các đặc trưng mới (Feature Engineering): Product Popularity, CLV, Seasonal trends, Marketing
3. Mã hóa biến phân loại và chuẩn hóa biến số
4. Phân chia Train/Test (Temporal vs Random Split) và xuất dữ liệu ra `data/processed/`

In [3]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse

# Tìm thư mục gốc dù Jupyter được mở từ thư mục dự án hay thư mục notebooks.
ROOT_DIR = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "README.md").exists()),
    Path.cwd().parent,
)
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src import config
from src.data.builder import build_product_week_panel
from src.data.feature_engineering import engineer_product_week_features
from src.data.preprocessing import build_weekly_preprocessor, split_product_week_data

## 1. Tải dữ liệu thô (Raw Data)

In [4]:
# Tạo hoặc nạp panel Sản phẩm–Tuần (tuần bắt đầu vào thứ Hai) từ các bảng Olist gốc.
if config.WEEKLY_PANEL_FILE.exists():
    weekly_panel = pd.read_csv(config.WEEKLY_PANEL_FILE, parse_dates=["week_start"])
else:
    weekly_panel = build_product_week_panel()

print(f"Product-week rows: {len(weekly_panel):,}")
print(f"Distinct products: {weekly_panel['product_id'].nunique():,}")
print(f"Calendar weeks: {weekly_panel['week_start'].nunique():,}")

Product-week rows: 1,262,576
Distinct products: 32,216
Calendar weeks: 103


## 2. Tiền xử lý dữ liệu (Missing Values & Outliers)

In [5]:
# Kiểm tra giá trị khuyết trước khi huấn luyện. Không cắt ngưỡng doanh thu vì đây
# là biến mục tiêu thực tế và cần được giữ nguyên phân phối.
missing_report = weekly_panel.isna().sum().sort_values(ascending=False)
display(missing_report[missing_report.gt(0)].rename("missing_count").to_frame())
print(f"Target minimum: {weekly_panel[config.WEEKLY_TARGET_COLUMN].min():.2f} BRL")
print(f"Target maximum: {weekly_panel[config.WEEKLY_TARGET_COLUMN].max():.2f} BRL")
print("Không áp dụng capping ngoại lai.")

,missing_count
avg_price_last_observed,32216


Target minimum: 0.00 BRL
Target maximum: 16200.00 BRL
Không áp dụng capping ngoại lai.


## 3. Tạo các đặc trưng mới (Feature Engineering theo đề bài)
- **Product Popularity**: reviews, ratings, popularity score
- **Customer Lifetime Value (CLV)**: RFM, AOV
- **Seasonal Trends**: month, quarter, holidays, peak season
- **Marketing**: ad spend ratio, discount rate

In [6]:
# Đặc trưng trễ, trượt, đánh giá và lịch sử khách hàng chỉ dùng các tuần đã hoàn tất.
featured = engineer_product_week_features(weekly_panel)
featured.to_csv(config.FEATURED_WEEKLY_PANEL_FILE, index=False)
feature_columns = config.WEEKLY_NUMERICAL_FEATURES + config.WEEKLY_CATEGORICAL_FEATURES
print(f"Featured panel shape: {featured.shape}")
print(f"Numeric predictors: {len(config.WEEKLY_NUMERICAL_FEATURES)}")
print(f"Categorical predictors before encoding: {len(config.WEEKLY_CATEGORICAL_FEATURES)}")
display(featured[feature_columns + [config.WEEKLY_TARGET_COLUMN]].head())

Featured panel shape: (1262576, 26)
Numeric predictors: 16
Categorical predictors before encoding: 1


,avg_price_last_observed,sales_lag_1,sales_lag_2,sales_lag_4,sales_rolling_mean_4,sales_rolling_std_4,units_lag_1,product_age_weeks,reviews_count_prior,rating_mean_prior,cumulative_customer_spend,purchase_frequency_prior,geographic_dispersion_prior,month,week_of_year,is_peak_shopping_season,category,sales
0,NaN,0.00,0.00,0.00,0.000000,0.000000,0.0,0,0.0,NaN,0.00,0.0,0.0,5,20,0,perfumery,101.65
1,101.65,101.65,0.00,0.00,50.825000,71.877404,1.0,1,0.0,NaN,101.65,1.0,1.0,5,21,0,perfumery,0.00
2,101.65,0.00,101.65,0.00,33.883333,58.687655,0.0,2,0.0,NaN,101.65,1.0,1.0,5,22,0,perfumery,0.00
3,101.65,0.00,0.00,0.00,25.412500,50.825000,0.0,3,0.0,NaN,101.65,1.0,1.0,6,23,0,perfumery,0.00
4,101.65,0.00,0.00,101.65,25.412500,50.825000,0.0,4,1.0,5.0,101.65,1.0,1.0,6,24,0,perfumery,0.00


## 4. Phân chia Train/Test Split (Tránh rò rỉ dữ liệu)

In [7]:
# Chia theo tuần lịch hoàn chỉnh để mọi sản phẩm trong cùng tuần thuộc cùng một tập.
splits = split_product_week_data(featured)
for split_name, split_frame in splits.items():
    print(
        f"{split_name:>10}: {len(split_frame):,} rows | "
        f"{split_frame['week_start'].min():%Y-%m-%d} to "
        f"{split_frame['week_start'].max():%Y-%m-%d}"
    )

assert splits["train"]["week_start"].max() < splits["validation"]["week_start"].min()
assert splits["validation"]["week_start"].max() < splits["test"]["week_start"].min()

     train: 1,016,290 rows | 2016-09-12 to 2018-07-02
validation: 150,060 rows | 2018-07-09 to 2018-08-06
      test: 96,226 rows | 2018-08-13 to 2018-08-27


## 5. Lưu dữ liệu đã xử lý ra `data/processed/`

In [8]:
# Chỉ học tham số điền khuyết và chuẩn hóa từ dữ liệu train trong quá khứ.
# Danh mục sản phẩm cung cấp 73 nhãn Olist; đây là siêu dữ liệu cấu trúc, không phải thống kê được fit.
known_categories = sorted(weekly_panel["category"].dropna().unique().tolist())
preprocessor = build_weekly_preprocessor(category_values=known_categories)
X_train_raw = splits["train"][feature_columns]
X_validation_raw = splits["validation"][feature_columns]
X_test_raw = splits["test"][feature_columns]
y_train = splits["train"][config.WEEKLY_TARGET_COLUMN].to_numpy(dtype=np.float32)
y_validation = splits["validation"][config.WEEKLY_TARGET_COLUMN].to_numpy(dtype=np.float32)
y_test = splits["test"][config.WEEKLY_TARGET_COLUMN].to_numpy(dtype=np.float32)

X_train = preprocessor.fit_transform(X_train_raw)
X_validation = preprocessor.transform(X_validation_raw)
X_test = preprocessor.transform(X_test_raw)

# Lưu ma trận sparse để tối ưu dung lượng cho dữ liệu one-hot hơn một triệu dòng.
for cache_path, matrix in [
    (config.CACHE_X_TRAIN, X_train),
    (config.CACHE_X_VAL, X_validation),
    (config.CACHE_X_TEST, X_test),
]:
    sparse.save_npz(cache_path, sparse.csr_matrix(matrix))
np.save(config.CACHE_Y_TRAIN, y_train)
np.save(config.CACHE_Y_VAL, y_validation)
np.save(config.CACHE_Y_TEST, y_test)
feature_names = preprocessor.get_feature_names_out().tolist()
config.CACHE_FEATURE_NAMES.write_text(json.dumps(feature_names, indent=2), encoding="utf-8")
joblib.dump(preprocessor, config.NOTEBOOK_SAVED_MODELS_DIR / "weekly_preprocessor.joblib")

print(f"Train matrix: {X_train.shape}; validation: {X_validation.shape}; test: {X_test.shape}")
print(f"Encoded predictor count: {len(feature_names)}")
print(f"Processed artifacts saved under: {config.PROCESSED_DATA_DIR}")

Train matrix: (1016290, 90); validation: (150060, 90); test: (96226, 90)
Encoded predictor count: 90
Processed artifacts saved under: E:\MachineLearning\sales_predictor-main\sales_predictor-main\data\processed
